# Group - DBLP Data Science Project
**Members:** Santiago Reyes (2174589), Ricardo Trevizo (2173424), Aqeel Somani (2364462)

## Step 1: Import Dataset and Turn to a DataFrame

In [1]:
import os
import glob
import pandas as pd

CACHE_PATH = "dblp_combined.parquet"
data_dir = os.path.join("dblp_data", "dblp-ref")
json_files = sorted(glob.glob(os.path.join(data_dir, "dblp-ref-*.json")))

if not json_files and not os.path.exists(CACHE_PATH):
    raise FileNotFoundError("No JSON files found in dblp_data/dblp-ref and no cache file found.")

if os.path.exists(CACHE_PATH):
    print("Loading from parquet cache...")
    combined_df = pd.read_parquet(CACHE_PATH)
else:
    print(f"Reading {len(json_files)} JSON files...")
    df_list = [pd.read_json(file, lines=True) for file in json_files]
    combined_df = pd.concat(df_list, ignore_index=True)
    combined_df.to_parquet(CACHE_PATH, index=False)
    print(f"Saved cache to {CACHE_PATH}")

print(f"Loaded {len(combined_df):,} records")
display(combined_df.head())

Loading from parquet cache...
Loaded 3,079,007 records


,abstract,authors,n_citation,references,title,venue,year,id
0,The purpose of this study is to develop a lear...,"[Makoto Satoh, Ryo Muramatsu, Mizue Kayama, Ka...",0,"[51c7e02e-f5ed-431a-8cf5-f761f266d4be, 69b625b...",Preliminary Design of a Network Protocol Learn...,international conference on human-computer int...,2013,00127ee2-cb05-48ce-bc49-9de556b93346
1,This paper describes the design and implementa...,"[Gareth Beale, Graeme Earl]",50,"[10482dd3-4642-4193-842f-85f3b70fcf65, 3133714...",A methodology for the physically accurate visu...,visual analytics science and technology,2011,001c58d3-26ad-46b3-ab3a-c1e557d16821
2,This article applied GARCH model instead AR or...,"[Altaf Hossain, Faisal Zaman, Mohammed Nasser,...",50,"[2d84c0f2-e656-4ce7-b018-90eda1c132fe, a083a1b...","Comparison of GARCH, Neural Network and Suppor...",pattern recognition and machine intelligence,2009,001c8744-73c4-4b04-9364-22d31a10dbf1
3,NaN,"[Jea-Bum Park, Byungmok Kim, Jian Shen, Sun-Yo...",0,"[8c78e4b0-632b-4293-b491-85b1976675e6, 9cdc54f...",Development of Remote Monitoring and Control D...,,2011,00338203-9eb3-40c5-9f31-cbac73a519ec
4,NaN,"[Giovanna Guerrini, Isabella Merlo]",2,None,Reasonig about Set-Oriented Methods in Object ...,,1998,0040b022-1472-4f70-a753-74832df65266


## Step 2: Find Missing Values

In [3]:
# Show missing values as a percentage table
def missing_values_table(df):
    mis_val = df.isnull().sum()
    mis_val_percent = 100 * df.isnull().sum() / len(df)
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
    mis_val_table_ren_columns = mis_val_table.rename(
        columns={0: 'Missing Values', 1: '% of Total Values'})
    return mis_val_table_ren_columns

table = missing_values_table(combined_df)
table.round(2)

,Missing Values,% of Total Values
abstract,530475,17.23
authors,4,0.00
n_citation,0,0.00
references,362865,11.79
title,0,0.00
venue,0,0.00
year,0,0.00
id,0,0.00


---
# Data Preprocessing
The following steps clean and prepare the dataset for EDA, Clustering, and Classification.

**Summary of steps:**
1. Fill missing values with safe defaults
2. Normalize and clean text (title + abstract)
3. Filter noisy venues with too few papers
4. Engineer new numerical features from existing columns
5. Impute numeric missing values with medians and apply Z-score normalization
6. TF-IDF vectorization of text
7. Dimensionality reduction with TruncatedSVD (for EDA scatter plots)

### Preprocessing Step 1: Fill Missing Values
- `abstract`: fill with empty string `""` so TF-IDF still works on these rows
- `references` and `authors`: fill with empty list `[]` since they are list columns
- We also add a flag column `has_abstract` so we can choose to exclude these rows in NLP tasks if needed

In [4]:
# Fill missing abstracts with empty string
combined_df['abstract'] = combined_df['abstract'].fillna('')

# Fill missing references and authors with empty lists
# NOTE: parquet reloads list columns as numpy arrays, not Python lists.
# Missing entries can appear as a float or None — both must be handled.
combined_df['references'] = combined_df['references'].apply(
    lambda x: [] if x is None or isinstance(x, float) else list(x)
)
combined_df['authors'] = combined_df['authors'].apply(
    lambda x: [] if x is None or isinstance(x, float) else list(x)
)

# Flag rows that have an abstract vs those that don't
combined_df['has_abstract'] = combined_df['abstract'] != ''

print(f"Total papers: {len(combined_df):,}")
print(f"Papers WITH abstract: {combined_df['has_abstract'].sum():,}")
print(f"Papers WITHOUT abstract: {(~combined_df['has_abstract']).sum():,}")
print("\nRemaining missing values:")
print(combined_df.isnull().sum())

Total papers: 3,079,007
Papers WITH abstract: 2,548,532
Papers WITHOUT abstract: 530,475

Remaining NaN values:
abstract        0
authors         0
n_citation      0
references      0
title           0
venue           0
year            0
id              0
has_abstract    0
dtype: int64


### Preprocessing Step 2: Normalize and Clean Text
We clean the `title` and `abstract` columns by:
- Converting to lowercase
- Removing special characters (keeping only letters, numbers, spaces)
- Normalizing extra whitespace

We then combine `clean_title` + `clean_abstract` into a single `text_combined` column that will be used for TF-IDF. Combining both gives the model a richer signal about each paper's topic.

In [5]:
import re
import pandas as pd

if "combined_df_filtered" not in globals():
    MIN_PAPERS_PER_VENUE = 50
    venue_counts_tmp = combined_df["venue"].value_counts()
    valid_venues_tmp = venue_counts_tmp[venue_counts_tmp >= MIN_PAPERS_PER_VENUE].index
    combined_df_filtered = combined_df[combined_df["venue"].isin(valid_venues_tmp)].copy()
    print(f"Created combined_df_filtered with {len(combined_df_filtered):,} rows for cleaning")

def clean_column_in_chunks(df, col_name, chunk_size=50_000):
    cleaned_parts = []
    total = len(df)
    for start in range(0, total, chunk_size):
        end = min(start + chunk_size, total)
        chunk = df[col_name].iloc[start:end].fillna("").astype("string[python]")
        chunk = chunk.str.lower()
        chunk = chunk.str.replace(r"[^a-z0-9\s]", " ", regex=True)
        chunk = chunk.str.replace(r"\s+", " ", regex=True)
        chunk = chunk.str.strip().fillna("")
        cleaned_parts.append(chunk)
    return pd.concat(cleaned_parts)

combined_df_filtered["clean_title"] = clean_column_in_chunks(combined_df_filtered, "title")
combined_df_filtered["clean_abstract"] = clean_column_in_chunks(combined_df_filtered, "abstract")
combined_df_filtered["text_combined"] = (
    combined_df_filtered["clean_title"] + " " + combined_df_filtered["clean_abstract"]
).str.strip()

print("Sample cleaned title:")
print(combined_df_filtered["clean_title"].iloc[0])
print("\nSample combined text (first 200 chars):")
print(combined_df_filtered["text_combined"].iloc[0][:200])


Created combined_df_filtered with 3,063,204 rows for cleaning
Sample cleaned title:
preliminary design of a network protocol learning tool based on the comprehension of high school students design by an empirical study using a simple mind map

Sample combined text (first 200 chars):
preliminary design of a network protocol learning tool based on the comprehension of high school students design by an empirical study using a simple mind map the purpose of this study is to develop a


### Preprocessing Step 3: Filter Noisy Venues
Venues with very few papers don't have enough signal for meaningful clustering or classification.
We keep only venues with **at least 50 papers** as our threshold.

This follows the rubric guideline: *"filter out noisy data points (e.g., venues with few papers) or select good data points (e.g., well known venues with sufficient papers)"*

In [6]:
# Count papers per venue
venue_counts = combined_df['venue'].value_counts()

print(f"Total unique venues before filtering: {len(venue_counts):,}")
print(f"\nVenue size distribution:")
print(venue_counts.describe())

# Keep only venues with at least 50 papers
MIN_PAPERS_PER_VENUE = 50
valid_venues = venue_counts[venue_counts >= MIN_PAPERS_PER_VENUE].index
if "combined_df_filtered" not in globals():
    combined_df_filtered = combined_df[combined_df['venue'].isin(valid_venues)].copy()

print(f"\nVenues kept (>= {MIN_PAPERS_PER_VENUE} papers): {len(valid_venues):,}")
print(f"Papers before filtering: {len(combined_df):,}")
print(f"Papers after filtering:  {len(combined_df_filtered):,}")
print(f"Papers removed: {len(combined_df) - len(combined_df_filtered):,}")

Total unique venues before filtering: 5,079

Venue size distribution:
count      5079.000000
mean        606.223075
std        7241.213946
min           1.000000
25%           2.000000
50%          71.000000
75%         447.000000
max      506699.000000
Name: count, dtype: float64

Venues kept (>= 50 papers): 2,673
Papers before filtering: 3,079,007
Papers after filtering:  3,063,204
Papers removed: 15,803


### Preprocessing Step 4: Engineer New Numerical Features
The dataset has no built-in numerical features beyond `year` and `n_citation`.
We derive new ones from existing columns to enrich our EDA and models.

In [7]:
# Number of authors on each paper
combined_df_filtered['author_count'] = combined_df_filtered['authors'].apply(len)

# Number of references each paper cites
combined_df_filtered['reference_count'] = combined_df_filtered['references'].apply(len)

# Word count of the abstract
combined_df_filtered['abstract_length'] = combined_df_filtered['clean_abstract'].apply(
    lambda x: len(x.split()) if x != '' else 0
)

# Decade bucket (useful for temporal EDA)
combined_df_filtered['decade'] = (combined_df_filtered['year'] // 10) * 10

# Summary of engineered features
print("New engineered features summary:")
display(combined_df_filtered[['author_count', 'reference_count', 'abstract_length', 'decade']].describe().round(2))

New engineered features summary:


,author_count,reference_count,abstract_length,decade
count,3063204.00,3063204.00,3063204.00,3063204.00
mean,3.08,8.17,120.03,2003.33
std,1.78,9.65,79.94,8.30
min,0.00,0.00,0.00,1930.00
25%,2.00,1.00,73.00,2000.00
50%,3.00,6.00,124.00,2010.00
75%,4.00,12.00,167.00,2010.00
max,351.00,1532.00,3575.00,2010.00


In [8]:
# Preprocessing Step 4.5: Numeric imputation + Z-score normalization
numeric_cols = ["year", "n_citation", "author_count", "reference_count", "abstract_length"]
numeric_cols = [c for c in numeric_cols if c in combined_df_filtered.columns]

missing_before = combined_df_filtered[numeric_cols].isnull().sum()

# Median imputation for numeric features
for col in numeric_cols:
    median_value = combined_df_filtered[col].median()
    combined_df_filtered[col] = combined_df_filtered[col].fillna(median_value)

missing_after = combined_df_filtered[numeric_cols].isnull().sum()

# Z-score normalization in new columns to preserve original scales
zscore_cols = []
for col in numeric_cols:
    col_std = combined_df_filtered[col].std(ddof=0)
    if pd.isna(col_std) or col_std == 0:
        print(f"Skipping {col}: zero variance")
        continue
    z_col = f"{col}_z"
    combined_df_filtered[z_col] = (combined_df_filtered[col] - combined_df_filtered[col].mean()) / col_std
    zscore_cols.append(z_col)

print("Missing values before median imputation:")
print(missing_before)
print("\nMissing values after median imputation:")
print(missing_after)
print(f"\nCreated z-score columns: {zscore_cols}")
if zscore_cols:
    display(combined_df_filtered[zscore_cols].describe().round(2))
else:
    print("No z-score columns were created.")

Missing values before median imputation:
year               0
n_citation         0
author_count       0
reference_count    0
abstract_length    0
dtype: int64

Missing values after median imputation:
year               0
n_citation         0
author_count       0
reference_count    0
abstract_length    0
dtype: int64

Created z-score columns: ['year_z', 'n_citation_z', 'author_count_z', 'reference_count_z', 'abstract_length_z']


,year_z,n_citation_z,author_count_z,reference_count_z,abstract_length_z
count,3063204.00,3063204.00,3063204.00,3063204.00,3063204.00
mean,-0.00,0.00,0.00,0.00,0.00
std,1.00,1.00,1.00,1.00,1.00
min,-9.18,-0.22,-1.73,-0.85,-1.50
25%,-0.48,-0.22,-0.61,-0.74,-0.59
50%,0.29,-0.15,-0.04,-0.23,0.05
75%,0.67,0.09,0.52,0.40,0.59
max,1.31,468.05,195.66,157.86,43.22


### Preprocessing Step 5: TF-IDF Vectorization
We convert the `text_combined` column into numerical features using TF-IDF.

- `max_features=5000`: keeps the top 5,000 most informative words to avoid a huge sparse matrix
- `stop_words='english'`: removes common words like 'the', 'and', 'is' that don't help distinguish topics
- `min_df=5`: a word must appear in at least 5 papers (removes typos and rare one-off words)
- `max_df=0.85`: ignores words that appear in more than 85% of papers (too common to be useful)

We use papers **with abstracts only** for the best quality TF-IDF features.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Set to a number (e.g. 100_000) for fast dev runs; None = automatic behavior below
DEV_SAMPLE = None

# Safety guard to avoid memory crashes on very large corpora
AUTO_MAX_TFIDF_ROWS = 300_000

has_abstract_mask = combined_df_filtered["has_abstract"].to_numpy(dtype=bool)
eligible_idx = np.flatnonzero(has_abstract_mask)

if DEV_SAMPLE is not None:
    target_n = min(DEV_SAMPLE, len(eligible_idx))
elif len(eligible_idx) > AUTO_MAX_TFIDF_ROWS:
    target_n = AUTO_MAX_TFIDF_ROWS
    print(f"[AUTO] Sampling {target_n:,} papers to prevent memory overflow")
else:
    target_n = len(eligible_idx)

if target_n < len(eligible_idx):
    rng = np.random.default_rng(42)
    chosen_idx = np.sort(rng.choice(eligible_idx, size=target_n, replace=False))
else:
    chosen_idx = eligible_idx

text_data = combined_df_filtered["text_combined"].iloc[chosen_idx].astype(str).reset_index(drop=True)
df_with_abstract = pd.DataFrame({"text_combined": text_data})

print(f"Papers used for TF-IDF (have abstract): {len(df_with_abstract):,}")

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    min_df=5,
    max_df=0.85,
    token_pattern=r'(?u)\b[a-z]{2,}\b'  # letters only, 2+ chars — excludes pure number tokens
)

X_tfidf = tfidf_vectorizer.fit_transform(df_with_abstract["text_combined"])

print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"  - Rows: {X_tfidf.shape[0]:,} papers")
print(f"  - Columns: {X_tfidf.shape[1]:,} word features")

feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"Sample TF-IDF features (words): {list(feature_names[:20])}")

[AUTO] Sampling 300,000 papers to prevent memory overflow
Papers used for TF-IDF (have abstract): 300,000
TF-IDF matrix shape: (300000, 5000)
  - Rows: 300,000 papers
  - Columns: 5,000 word features
Sample TF-IDF features (words): ['abilities', 'ability', 'able', 'abnormal', 'absence', 'absolute', 'abstract', 'abstraction', 'abstractions', 'ac', 'academic', 'accelerate', 'accelerated', 'acceleration', 'accelerator', 'acceptable', 'acceptance', 'accepted', 'access', 'accessed']


### Preprocessing Summary
| Step | Action | Result |
|------|--------|--------|
| 1 | Fill missing values (abstract -> "", refs/authors -> `[]`) | No more missing values in key text/list fields |
| 2 | Clean text (lowercase, special chars, whitespace) | `clean_title`, `clean_abstract`, `text_combined` |
| 3 | Filter venues with < 50 papers | Removed noisy/insignificant venues |
| 4 | Engineer numerical features | `author_count`, `reference_count`, `abstract_length`, `decade` |
| 5 | Median imputation + Z-score normalization on numeric features | Added scaled columns like `year_z`, `n_citation_z` |
| 6 | TF-IDF vectorization (5000 features) | `X_tfidf` sparse matrix ready for clustering/classification |

In [ ]:
import joblib

# Save the cleaned/filtered DataFrame for EDA and classification notebooks
combined_df_filtered.to_parquet("dblp_preprocessed.parquet", index=False)

# Save the TF-IDF matrix and vectorizer so no need to recompute
joblib.dump(X_tfidf, "tfidf_matrix.pkl")
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")

print("Saved:")
print(f"  dblp_preprocessed.parquet  — {len(combined_df_filtered):,} rows, {combined_df_filtered.shape[1]} columns")
print(f"  tfidf_matrix.pkl           — {X_tfidf.shape[0]:,} papers x {X_tfidf.shape[1]:,} features")
print(f"  tfidf_vectorizer.pkl       — fitted TfidfVectorizer")